In [ ]:
import numpy as np
import pandas as pd
import time
from scipy.interpolate import LinearNDInterpolator, RegularGridInterpolator, SmoothBivariateSpline
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt

# ===============================================================
# 1. Load the dataset
# ===============================================================
data = [
(0,0,149.51247930624),(0,0.05,477.073388169435),(0,0.1,1489.96400273872),
(0,0.15,2502.854617308),(0,0.2,3516.88581656464),(0,0.25,4531.17546666299),
(0,0.3,5352.01006901315),(0,0.35,5880.59479344536),(0,0.4,6409.17951787757),
(0,0.45,6769.40385235567),(0,0.5,7126.39616686231),(0,0.55,7479.89901017385),
(0,0.6,7830.69947359865),(0,0.65,8156.10202678075),(0,0.7,8332.98280809141),
(0,0.75,8509.86358940208),(0,0.8,8091.31040704162),(0,0.85,7450.95860235664),
(0,0.9,6810.60679767167),(0,0.95,6170.25499298669),(0,1,5529.90318830172),
(10,0,197.995233619588),(10,0.05,1520.30548979673),(10,0.1,4101.16078819117),
(10,0.15,6603.95999401886),(10,0.2,8932.31003901793),(10,0.25,11017.3594380417),
(10,0.3,11545.4494024806),(10,0.35,12073.5393669196),(10,0.4,11998.196878099),
(10,0.45,11898.4439872119),(10,0.5,12081.8593685321),(10,0.55,12341.9094445186),
(10,0.6,12734.6218070063),(10,0.65,13211.0982375843),(10,0.7,13697.0258566703),
(10,0.75,14195.0299020275),(10,0.8,14134.8759026643),(10,0.85,12526.7326739401),
(10,0.9,10918.5894452159),(10,0.95,9310.44621649165),(10,1,7702.30298776745),
(20,0,329.501985232743),(20,0.05,3886.20453837969),(20,0.1,7612.96848363861),
(20,0.15,11660.5374403703),(20,0.2,15236.7246277382),(20,0.25,17644.323838668),
(20,0.3,19474.7163003259),(20,0.35,19355.5634004475),(20,0.4,19085.070609186),
(20,0.45,18073.9761633465),(20,0.5,17148.7692483824),(20,0.55,16914.6328489478),
(20,0.6,16771.9761982571),(20,0.65,18307.5875270115),(20,0.7,19843.1988557658),
(20,0.75,20059.7089253373),(20,0.8,20266.3287063771),(20,0.85,17687.9259753949),
(20,0.9,14911.1625729355),(20,0.95,12134.3991704761),(20,1,9357.63576801671),
(30,0,4083.28436564144),(30,0.05,8469.27591324817),(30,0.1,12855.2674608549),
(30,0.15,16335.3126104586),(30,0.2,19802.4274714701),(30,0.25,23340.4275859378),
(30,0.3,26880.490604979),(30,0.35,26287.7608874102),(30,0.4,25512.0900278903),
(30,0.45,23879.2003222203),(30,0.5,22195.2176491575),(30,0.55,21258.7633670801),
(30,0.6,20378.7694141054),(30,0.65,25098.5882935986),(30,0.7,30332.7427562444),
(30,0.75,27565.2024906821),(30,0.8,23925.6282475197),(30,0.85,20589.1410241325),
(30,0.9,17291.0806870524),(30,0.95,13993.0203499723),(30,1,10694.9600128923),
(40,0,12500),(40,0.05,14778.3313432087),(40,0.1,16828.6235440446),
(40,0.15,19003.1542978902),(40,0.2,21416.4983826924),(40,0.25,24179.8718572875),
(40,0.3,27481.7735275019),(40,0.35,29140.7554844248),(40,0.4,28755.8316781002),
(40,0.45,27795.9918197639),(40,0.5,26255.5133715658),(40,0.55,25115.1561833649),
(40,0.6,24303.2503930983),(40,0.65,30199.3417699096),(40,0.7,40547.7811965492),
(40,0.75,37546.7950266708),(40,0.8,27444.1857043381),(40,0.85,22034.5605473862),
(40,0.9,18592.9349660142),(40,0.95,15151.3093846421),(40,1,11709.6838032701)
]
df = pd.DataFrame(data, columns=["temp", "soc", "c"])

temps = df["temp"].to_numpy()
socs = df["soc"].to_numpy()
C = df["c"].to_numpy()

# ===============================================================
# 2. Build models
# ===============================================================
# Reference: current class
ref_model = LinearNDInterpolator(list(zip(temps, socs)), C)

# Dense lookup (regular grid)
temp_grid = np.unique(temps)
soc_grid = np.unique(socs)
C_grid = np.zeros((len(temp_grid), len(soc_grid)))
for i, t in enumerate(temp_grid):
    mask = df["temp"] == t
    C_grid[i, :] = df[mask].sort_values("soc")["c"].values
lookup_model = RegularGridInterpolator((temp_grid, soc_grid), C_grid)

# Dense augmented model (Smooth Bivariate Spline)
# First create dense augmented data using the reference interpolator
t_dense = np.linspace(0, 40, 81)
s_dense = np.linspace(0, 1, 201)
T, S = np.meshgrid(t_dense, s_dense)
C_dense = ref_model(T, S)
mask = np.isnan(C_dense)
if np.any(mask):
    C_dense[mask] = np.nanmean(C_dense)
model_spline = SmoothBivariateSpline(T.ravel(), S.ravel(), C_dense.ravel(), s=0)

# ===============================================================
# 3. Simulation inputs
# ===============================================================
N = 50000  # number of random test points
temp_test = np.random.uniform(0, 40, N)
soc_test = np.random.uniform(0, 1, N)
points = np.column_stack((temp_test, soc_test))

# ===============================================================
# 4. Benchmark each method
# ===============================================================
def benchmark(func, name, reference_values):
    start = time.perf_counter()
    preds = np.array([func(p) for p in points])
    elapsed = time.perf_counter() - start
    rmse = np.sqrt(mean_squared_error(reference_values, preds))
    print(f"{name:25s}  time = {elapsed:.3f}s  avg/call = {elapsed/N*1e6:7.2f} µs  RMSE = {rmse:.4f}")
    return preds, elapsed, rmse

# reference ("ground truth") values from original interpolator
ref_values = np.array([ref_model(p) for p in points])

# Dense lookup interpolator
lookup_values, t_lookup, e_lookup = benchmark(lookup_model, "Dense lookup", ref_values)

# Spline model
spline_values, t_spline, e_spline = benchmark(lambda p: model_spline(p[0], p[1])[0][0], "Spline surrogate", ref_values)

# ===============================================================
# 5. Summary
# ===============================================================
print("\n--- Summary ---")
print(f"Reference interpolator (LinearND): {N/t_lookup:.1f} calls/sec baseline = 1.0x")
print(f"Dense lookup speedup:              {t_lookup/t_lookup:.2f}x (baseline)")
print(f"Spline surrogate speedup:          {t_lookup/t_spline:.2f}x")

# ===============================================================
# 6. Optional plots
# ===============================================================
plt.hist(ref_values - lookup_values, bins=50, alpha=0.6, label="Dense lookup error")
plt.hist(ref_values - spline_values, bins=50, alpha=0.6, label="Spline error")
plt.legend()
plt.xlabel("Error (model - reference)")
plt.ylabel("Frequency")
plt.title("Error Distribution Comparison")
plt.show()
